# CSCN8020 Assignment 1 - Reinforcement Learning Programming

### Student name: Anthony Izevbokun  
### Student ID: 9016626  
### Course: CSCN8020 
### Repository: `https://github.com/chooksemmanuel/CSCN8020_Assignment1.git`



## Assignment assumptions and reading notes

This assignment covers four reinforcement learning problems: designing a pick-and-place robot as an MDP, performing two manual value-iteration sweeps on a 2x2 Gridworld, running value iteration and in-place value iteration on a 5x5 Gridworld, and applying off-policy Monte Carlo control with importance sampling on the same 5x5 environment.

The following assumptions are applied consistently throughout:

1. **Discount factor:** \(\gamma = 0.9\), applied uniformly across all gridworld problems as recommended by the Monte Carlo task description.
2. **Reward convention:** the reward signal is tied to the state the agent transitions into. Reaching the goal yields \(+10\), landing on a grey state yields \(-5\), and entering any ordinary state yields \(-1\).
3. **Wall collision convention:** when an action would take the agent out of bounds, it remains in its current cell and receives that cell's reward.
4. **Terminal state convention:** once the agent steps into the goal in the 5x5 grid, the episode concludes. The goal's value of \(+10\) represents the immediate reward, not compounded future returns.
5. **Action interpretation:** the problem statement lists `right, down, down, up` for the 5x5 grid actions. I treat the duplicate `down` as a typographical error and use the standard four-directional action set: `right`, `down`, `left`, `up`.

## Imports, logging, and reusable objects

The implementation follows an object-oriented structure. Each concern — environment dynamics, policy definitions, agent algorithms, and utility functions — is encapsulated in its own module under the `src/` directory.

Key classes used in this notebook:

- `DeterministicGridWorldEnv`: encapsulates the grid layout, reward function, and transition dynamics.
- `ValueIterationAgent`: implements synchronous (standard) value iteration.
- `InPlaceValueIterationAgent`: implements the in-place variant of value iteration.
- `OffPolicyMonteCarloAgent`: implements off-policy Monte Carlo control using weighted importance sampling.
- `RandomPolicy`: uniform random behavior policy used for Monte Carlo episode generation.
- `configure_logger`: sets up the execution log file required by the assignment.

In [1]:
import os
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

sys.path.append(os.getcwd())

from src.environments import create_2x2_gridworld, create_5x5_gridworld
from src.agents import ValueIterationAgent, InPlaceValueIterationAgent, OffPolicyMonteCarloAgent
from src.utils import configure_logger, values_to_dataframe, policy_to_dataframe

logger = configure_logger("logs/sample_execution.log")
np.set_printoptions(precision=3, suppress=True)

# Problem 1 - Pick-and-Place Robot MDP Design

A pick-and-place robot cycles through a repeated task: retrieve an object from a fixed pickup location and deposit it at a designated target. The goal is to learn motor commands that produce **fast and smooth** motion, so the agent must operate at the level of joint control rather than abstract waypoints.

## MDP definition

An MDP is formally described by the tuple:

\[
\mathcal{M} = (S, A, P, R, \gamma)
\]

where:

- \(S\) is the state space.
- \(A\) is the action space.
- \(P(s'|s,a)\) is the transition function.
- \(R(s,a,s')\) is the reward function.
- \(\gamma\) is the discount factor.

## State space

To make informed control decisions the agent needs positional and dynamic information about both the robot and the object:

\[
s_t = [q_t, \dot{q}_t, g_t, p_{object}, p_{target}, d_{gripper,object}, d_{object,target}, c_t]
\]

Where:

- \(q_t\): current joint angles of the robot arm.
- \(\dot{q}_t\): current joint angular velocities.
- \(g_t\): gripper status (open, closing, or grasping).
- \(p_{object}\): three-dimensional position of the target object.
- \(p_{target}\): position of the drop-off location.
- \(d_{gripper,object}\): Euclidean distance from end-effector to object.
- \(d_{object,target}\): distance from object to the placement zone.
- \(c_t\): proximity or collision indicator for safety monitoring.

Including velocity terms is essential because smooth motion requires awareness of how the robot is moving, not only where it currently is.

## Action space

The agent commands the robot at the actuator level using continuous torque or velocity signals:

\[
a_t = [\tau_1, \tau_2, ..., \tau_n, u_{gripper}]
\]

Where:

- \(\tau_i\): commanded torque or velocity for joint \(i\).
- \(u_{gripper}\): open or close signal for the gripper.

This formulation makes the problem a continuous-control reinforcement learning task.

## Transition model

The transition function captures the physical response to a given motor command:

\[
P(s_{t+1}|s_t,a_t)
\]

In practice, this transition reflects the robot's physical dynamics including inertia, joint friction, contact forces, and gripper mechanics. During training, a physics simulator provides these transitions before any transfer to hardware.

## Reward design

The reward function is shaped to simultaneously encourage task completion, execution speed, and kinematic smoothness:

\[
r_t = 100I_{placed} + 20I_{grasped} - d_{gripper,object} - d_{object,target} - 0.01\|a_t\|^2 - 0.05\|\Delta a_t\|^2 - 50I_{collision} - 1
\]

Each term serves a specific purpose:

- `+100` for successfully placing the object at the target.
- `+20` for achieving a stable grasp.
- Distance penalties continuously draw the robot toward the object and then the target.
- Action magnitude penalty reduces unnecessary force.
- Smoothness penalty on action changes discourages abrupt or jerky movements.
- Collision penalty enforces safe operation.
- Per-step penalty creates pressure for efficient execution.

## Talking points

**A. Key RL feature:** The central RL concept is the MDP structure itself. The robot observes its state, selects motor commands, receives a reward signal, and iteratively refines its policy — without any labelled demonstrations of correct behavior.

**B. Implementation/testing challenge:** Reward shaping is the hardest part. If the task-completion bonus dominates, the robot may succeed in reaching the goal but move erratically. If the smoothness penalty is weighted too heavily, the agent becomes overly cautious and slow. Balancing these terms requires empirical tuning.

**C. Why this is reinforcement learning:** This is RL because the agent learns through environment interaction over time. It receives no ground-truth labels for what the correct joint torques should be at each moment. Learning emerges from cumulative reward signals tied to state transitions and action consequences.

# Problem 2 - 2x2 Gridworld Manual Value Iteration

The grid layout and per-state rewards are:

| State | Reward |
|---|---:|
| \(s_1\) | 5 |
| \(s_2\) | 10 |
| \(s_3\) | 1 |
| \(s_4\) | 2 |

Spatial arrangement:

|  |  |
|---|---|
| \(s_1\) | \(s_2\) |
| \(s_3\) | \(s_4\) |

All four actions (`up`, `down`, `left`, `right`) are available in every state. Attempting an action that would move the agent off the grid leaves it in the current cell.

All state values are initialised to zero:

\[
V_0(s) = 0 \quad \text{for all states}
\]

with \(\gamma = 0.9\). Each update applies the Bellman optimality operator:

\[
V_{k+1}(s) = \max_a \left[R(s') + \gamma V_k(s')\right]
\]

where \(s'\) is the state reached after taking action \(a\) from \(s\).

## Iteration 1

Starting from zero-initialised values:

\[
V_0(s_1)=0, \quad V_0(s_2)=0, \quad V_0(s_3)=0, \quad V_0(s_4)=0
\]

### Update for \(s_1\)

Evaluating each action from \(s_1\):

- `up` hits wall: stay in \(s_1\), reward = 5
- `down` transitions to \(s_3\), reward = 1
- `left` hits wall: stay in \(s_1\), reward = 5
- `right` transitions to \(s_2\), reward = 10

\[
V_1(s_1)=\max(5+0.9(0),\ 1+0.9(0),\ 5+0.9(0),\ 10+0.9(0)) = 10
\]

### Update for \(s_2\)

Evaluating each action from \(s_2\):

- `up` hits wall: stay in \(s_2\), reward = 10
- `down` transitions to \(s_4\), reward = 2
- `left` transitions to \(s_1\), reward = 5
- `right` hits wall: stay in \(s_2\), reward = 10

\[
V_1(s_2)=\max(10,\ 2,\ 5,\ 10)=10
\]

### Update for \(s_3\)

Evaluating each action from \(s_3\):

- `up` transitions to \(s_1\), reward = 5
- `down` hits wall: stay in \(s_3\), reward = 1
- `left` hits wall: stay in \(s_3\), reward = 1
- `right` transitions to \(s_4\), reward = 2

\[
V_1(s_3)=\max(5,\ 1,\ 1,\ 2)=5
\]

### Update for \(s_4\)

Evaluating each action from \(s_4\):

- `up` transitions to \(s_2\), reward = 10
- `down` hits wall: stay in \(s_4\), reward = 2
- `left` transitions to \(s_3\), reward = 1
- `right` hits wall: stay in \(s_4\), reward = 2

\[
V_1(s_4)=\max(10,\ 2,\ 1,\ 2)=10
\]

Values after Iteration 1:

\[
V_1 =
\begin{bmatrix}
10 & 10 \\
5 & 10
\end{bmatrix}
\]

Greedy policy derived from \(V_1\):

\[
\pi_1(s_1)=\text{right}, \quad \pi_1(s_2)=\text{up/right}, \quad \pi_1(s_3)=\text{up}, \quad \pi_1(s_4)=\text{up}
\]

## Iteration 2

Using \(V_1\) as the starting point to compute \(V_2\):

### Update for \(s_1\)

\[
V_2(s_1)=\max(5+0.9(10),\ 1+0.9(5),\ 5+0.9(10),\ 10+0.9(10))
\]

\[
V_2(s_1)=\max(14,\ 5.5,\ 14,\ 19)=19
\]

### Update for \(s_2\)

\[
V_2(s_2)=\max(10+0.9(10),\ 2+0.9(10),\ 5+0.9(10),\ 10+0.9(10))
\]

\[
V_2(s_2)=\max(19,\ 11,\ 14,\ 19)=19
\]

### Update for \(s_3\)

\[
V_2(s_3)=\max(5+0.9(10),\ 1+0.9(5),\ 1+0.9(5),\ 2+0.9(10))
\]

\[
V_2(s_3)=\max(14,\ 5.5,\ 5.5,\ 11)=14
\]

### Update for \(s_4\)

\[
V_2(s_4)=\max(10+0.9(10),\ 2+0.9(10),\ 1+0.9(5),\ 2+0.9(10))
\]

\[
V_2(s_4)=\max(19,\ 11,\ 5.5,\ 11)=19
\]

Values after Iteration 2:

\[
V_2 =
\begin{bmatrix}
19 & 19 \\
14 & 19
\end{bmatrix}
\]

## Problem 2 code verification

The primary deliverable for Problem 2 is the manual step-by-step derivation shown above. To further validate those results, the following code cell runs the same two iterations programmatically and confirms the computed values align with the hand-calculated results.

In [2]:
grid2 = create_2x2_gridworld()
vi_2_iteration = ValueIterationAgent(
    env=grid2,
    gamma=0.9,
    theta=0.0,
    max_iterations=2,
    logger=logger
).run()

print("Value function after two iterations:")
display(values_to_dataframe(vi_2_iteration.values, 2, 2))

print("Greedy policy after two iterations:")
display(policy_to_dataframe(grid2.render_policy(vi_2_iteration.policy)))

2026-06-11 10:55:42,367 | INFO | Starting synchronous value iteration: gamma=0.9 theta=0.0
2026-06-11 10:55:42,405 | INFO | Value Iteration iteration=1 delta=10.00000000
2026-06-11 10:55:42,436 | INFO | Value Iteration iteration=2 delta=9.00000000
2026-06-11 10:55:42,439 | INFO | Finished synchronous value iteration in 2 iterations and 0.070863 seconds
2026-06-11 10:55:42,442 | INFO | Final values: [19.0, 19.0, 14.0, 19.0]


Value function after two iterations:


,col 0,col 1
row 0,19.0,19.0
row 1,14.0,19.0


Greedy policy after two iterations:


,col 0,col 1
row 0,→,→
row 1,↑,↑


## Problem 2 talking points

**A. Key RL feature:** The defining mechanism here is the Bellman optimality update. At each iteration, every state considers all available actions, evaluates the immediate reward plus discounted future value, and retains the maximum — driving the value function toward \(V^*\).

**B. Implementation/testing challenge:** The most important detail is the reward convention. Using the reward of the destination state (rather than the source) must be consistent between the manual derivation and the coded verification to produce matching results.

**C. Why this is reinforcement learning:** This is RL because the value of each state is defined through anticipated future agent-environment interaction. Rather than relying on labelled examples of correct decisions, the algorithm deduces optimal behaviour by propagating reward signals backwards through potential state transitions.

# Problem 3 - 5x5 Gridworld Value Iteration and In-Place Variation

The 5x5 Gridworld is configured as follows:

- Terminal goal state: \(s_{4,4}\)
- Penalty (grey) states: \(s_{2,2}\), \(s_{3,0}\), and \(s_{0,4}\)
- Goal reward: \(+10\)
- Grey-state reward: \(-5\)
- Standard step reward: \(-1\)

Both value iteration variants operate with full knowledge of the environment model, meaning the agent can directly evaluate the outcome of every action from every state without needing to generate episodes.

In [3]:
grid5 = create_5x5_gridworld()

print("Reward grid used for Problem 3 and Problem 4:")
display(values_to_dataframe(grid5.reward_grid().reshape(-1), 5, 5))

Reward grid used for Problem 3 and Problem 4:


,col 0,col 1,col 2,col 3,col 4
row 0,-1.0,-1.0,-1.0,-1.0,-5.0
row 1,-1.0,-1.0,-1.0,-1.0,-1.0
row 2,-1.0,-1.0,-5.0,-1.0,-1.0
row 3,-5.0,-1.0,-1.0,-1.0,-1.0
row 4,-1.0,-1.0,-1.0,-1.0,10.0


## Standard value iteration

Synchronous (standard) value iteration maintains two separate value arrays during each sweep:

- `values`: the value estimates carried over from the previous sweep.
- `new_values`: the updated estimates being computed in the current sweep.

By strictly reading from the old array and writing to the new one, this approach guarantees that all states within a single sweep are updated using only values from the same previous iteration.

In [4]:
standard_vi = ValueIterationAgent(
    env=grid5,
    gamma=0.9,
    theta=1e-6,
    max_iterations=1000,
    logger=logger
).run()

print(f"Synchronous value iteration converged in {standard_vi.iterations} iterations.")
print(f"Runtime: {standard_vi.elapsed_seconds:.6f} seconds")

print("Optimal state-value function V*:")
display(values_to_dataframe(standard_vi.values, 5, 5))

print("Optimal greedy policy π*:")
display(policy_to_dataframe(grid5.render_policy(standard_vi.policy)))

2026-06-11 10:55:42,636 | INFO | Starting synchronous value iteration: gamma=0.9 theta=1e-06
2026-06-11 10:55:42,641 | INFO | Value Iteration iteration=1 delta=10.00000000
2026-06-11 10:55:42,644 | INFO | Value Iteration iteration=2 delta=9.00000000
2026-06-11 10:55:42,648 | INFO | Value Iteration iteration=3 delta=8.10000000
2026-06-11 10:55:42,686 | INFO | Value Iteration iteration=4 delta=7.29000000
2026-06-11 10:55:42,700 | INFO | Value Iteration iteration=5 delta=6.56100000
2026-06-11 10:55:42,708 | INFO | Finished synchronous value iteration in 9 iterations and 0.069229 seconds
2026-06-11 10:55:42,711 | INFO | Final values: [-0.434, 0.629, 1.81, 3.122, 4.58, 0.629, 1.81, 3.122, 4.58, 6.2, 1.81, 3.122, 4.58, 6.2, 8.0, 3.122, 4.58, 6.2, 8.0, 10.0, 4.58, 6.2, 8.0, 10.0, 10.0]


Synchronous value iteration converged in 9 iterations.
Runtime: 0.069229 seconds
Optimal state-value function V*:


,col 0,col 1,col 2,col 3,col 4
row 0,-0.434,0.629,1.810,3.122,4.58
row 1,0.629,1.810,3.122,4.580,6.20
row 2,1.810,3.122,4.580,6.200,8.00
row 3,3.122,4.580,6.200,8.000,10.00
row 4,4.580,6.200,8.000,10.000,10.00


Optimal greedy policy π*:


,col 0,col 1,col 2,col 3,col 4
row 0,→,→,→,↓,↓
row 1,→,→,→,→,↓
row 2,→,↓,→,→,↓
row 3,→,→,→,→,↓
row 4,→,→,→,→,G


## In-place value iteration

The in-place variant operates with a single shared value array. Once a state's value is revised, the updated figure is immediately available to any subsequent state within the same sweep.

Because the updated values propagate within a single pass, this approach can reach convergence with fewer sweeps in some environments. However, for this deterministic gridworld, both methods are expected to converge to the same optimal value function and policy.

In [5]:
in_place_vi = InPlaceValueIterationAgent(
    env=grid5,
    gamma=0.9,
    theta=1e-6,
    max_iterations=1000,
    logger=logger
).run()

print(f"In-place value iteration converged in {in_place_vi.iterations} iterations.")
print(f"Runtime: {in_place_vi.elapsed_seconds:.6f} seconds")

print("In-place state-value function:")
display(values_to_dataframe(in_place_vi.values, 5, 5))

print("In-place greedy policy:")
display(policy_to_dataframe(grid5.render_policy(in_place_vi.policy)))

2026-06-11 10:55:42,925 | INFO | Starting in-place value iteration: gamma=0.9 theta=1e-06
2026-06-11 10:55:42,928 | INFO | In-place Value Iteration iteration=1 delta=10.00000000
2026-06-11 10:55:42,930 | INFO | In-place Value Iteration iteration=2 delta=9.00000000
2026-06-11 10:55:42,934 | INFO | In-place Value Iteration iteration=3 delta=8.10000000
2026-06-11 10:55:42,936 | INFO | In-place Value Iteration iteration=4 delta=7.29000000
2026-06-11 10:55:42,940 | INFO | In-place Value Iteration iteration=5 delta=6.56100000
2026-06-11 10:55:42,946 | INFO | Finished in-place value iteration in 9 iterations and 0.020210 seconds
2026-06-11 10:55:42,948 | INFO | Final in-place values: [-0.434, 0.629, 1.81, 3.122, 4.58, 0.629, 1.81, 3.122, 4.58, 6.2, 1.81, 3.122, 4.58, 6.2, 8.0, 3.122, 4.58, 6.2, 8.0, 10.0, 4.58, 6.2, 8.0, 10.0, 10.0]


In-place value iteration converged in 9 iterations.
Runtime: 0.020210 seconds
In-place state-value function:


,col 0,col 1,col 2,col 3,col 4
row 0,-0.434,0.629,1.810,3.122,4.58
row 1,0.629,1.810,3.122,4.580,6.20
row 2,1.810,3.122,4.580,6.200,8.00
row 3,3.122,4.580,6.200,8.000,10.00
row 4,4.580,6.200,8.000,10.000,10.00


In-place greedy policy:


,col 0,col 1,col 2,col 3,col 4
row 0,→,→,→,↓,↓
row 1,→,→,→,→,↓
row 2,→,↓,→,→,↓
row 3,→,→,→,→,↓
row 4,→,→,→,→,G


In [6]:
max_difference = np.max(np.abs(standard_vi.values - in_place_vi.values))
policy_match = np.array_equal(standard_vi.policy, in_place_vi.policy)

comparison_vi = pd.DataFrame({
    "Method": ["Synchronous Value Iteration", "In-Place Value Iteration"],
    "Iterations/Sweeps": [standard_vi.iterations, in_place_vi.iterations],
    "Runtime seconds": [standard_vi.elapsed_seconds, in_place_vi.elapsed_seconds],
    "Complexity": ["O(iterations × states × actions)", "O(iterations × states × actions)"],
})

display(comparison_vi)
print(f"Maximum absolute value difference: {max_difference:.8f}")
print(f"Do the policies match? {policy_match}")

,Method,Iterations/Sweeps,Runtime seconds,Complexity
0,Synchronous Value Iteration,9,0.069229,O(iterations × states × actions)
1,In-Place Value Iteration,9,0.020210,O(iterations × states × actions)


Maximum absolute value difference: 0.00000000
Do the policies match? True


## Problem 3 comments on performance and complexity

Both synchronous and in-place value iteration apply Bellman optimality backups across every state-action pair on each sweep. With 25 states and 4 actions, each sweep involves approximately \(25 \times 4 = 100\) backup operations.

The computational complexity of value iteration is:

\[
O(K|S||A|)
\]

where \(K\) is the number of sweeps, \(|S|\) is the number of states, and \(|A|\) is the number of actions.

Unlike Monte Carlo methods, value iteration does **not** generate episodes. Progress is measured in sweeps, so comparisons for Problem 3 are reported in terms of iterations rather than episode counts.

## Problem 3 talking points

**A. Key RL feature:** The central technique is value iteration. By repeatedly applying the Bellman optimality operator across all states, the algorithm converges to \(V^*\) and allows the greedy policy \(\pi^*\) to be extracted directly.

**B. Implementation/testing challenge:** The most delicate aspect was ensuring consistent handling of terminal and invalid states. The goal state must end further propagation, and wall collisions must return the agent to its current position — both rules must be encoded identically in the environment and relied upon by the agent.

**C. Why this is reinforcement learning:** This is RL because the algorithm derives an optimal control policy through the interaction between states, actions, rewards, transitions, and discounted future returns. No labelled action sequences are provided; the policy emerges purely from value estimates derived through dynamic programming.

# Problem 4 - Off-Policy Monte Carlo with Importance Sampling

Problem 4 revisits the 5x5 Gridworld, this time estimating values from sampled experience rather than sweeping through an exact environment model.

The method is **off-policy** because learning involves two distinct policies:

- **Behavior policy \(b(a|s)\):** a uniform random policy that generates exploratory episodes covering a wide range of state-action pairs.
- **Target policy \(\pi(a|s)\):** a greedy policy that is iteratively improved using the action-value estimates built from those episodes.

Returns are computed by processing each episode in reverse:

\[
G \leftarrow \gamma G + R_{t+1}
\]

An importance-sampling ratio adjusts for the discrepancy between the behavior and target distributions:

\[
W \leftarrow W \times \frac{1}{b(A_t|S_t)}
\]

Since the behavior policy is uniform across four actions:

\[
b(a|s)=0.25
\]

The algorithm implements weighted importance sampling as described in Sutton and Barto's off-policy Monte Carlo control pseudocode.

In [7]:
mc_agent = OffPolicyMonteCarloAgent(
    env=grid5,
    gamma=0.9,
    episodes=5000,
    max_steps_per_episode=100,
    seed=7,
    logger=logger
)
mc_result = mc_agent.run()

print(f"Off-policy Monte Carlo completed {mc_result.episodes} episodes.")
print(f"Runtime: {mc_result.elapsed_seconds:.6f} seconds")
print(f"Visited states with weighted updates: {mc_result.visited_state_count}")

print("Monte Carlo estimated state-value function:")
display(values_to_dataframe(mc_result.values, 5, 5))

print("Monte Carlo greedy target policy:")
display(policy_to_dataframe(grid5.render_policy(mc_result.policy)))

2026-06-11 10:55:43,147 | INFO | Starting off-policy Monte Carlo: episodes=5000 gamma=0.9
2026-06-11 10:55:43,151 | INFO | MC episode=1 length=100
2026-06-11 10:55:43,164 | INFO | MC episode=10 length=23
2026-06-11 10:55:43,233 | INFO | MC episode=100 length=100
2026-06-11 10:55:44,237 | INFO | MC episode=1000 length=55
2026-06-11 10:55:46,732 | INFO | MC episode=5000 length=1
2026-06-11 10:55:46,734 | INFO | Finished off-policy MC in 3.586793 seconds
2026-06-11 10:55:46,736 | INFO | MC estimated values: [-0.438, 0.62, 1.764, 3.023, 4.495, 0.484, 1.723, 3.046, 4.533, 6.13, 1.708, 3.04, 4.492, 6.129, 7.964, 3.085, 4.533, 6.153, 7.971, 10.0, 4.532, 5.895, 7.962, 10.0, 10.0]


Off-policy Monte Carlo completed 5000 episodes.
Runtime: 3.586793 seconds
Visited states with weighted updates: 24
Monte Carlo estimated state-value function:


,col 0,col 1,col 2,col 3,col 4
row 0,-0.438,0.620,1.764,3.023,4.495
row 1,0.484,1.723,3.046,4.533,6.130
row 2,1.708,3.040,4.492,6.129,7.964
row 3,3.085,4.533,6.153,7.971,10.000
row 4,4.532,5.895,7.962,10.000,10.000


Monte Carlo greedy target policy:


,col 0,col 1,col 2,col 3,col 4
row 0,→,↓,↓,↓,↓
row 1,→,→,→,↓,↓
row 2,→,↓,→,→,↓
row 3,↓,→,→,↓,↓
row 4,→,→,→,→,G


In [8]:
absolute_errors = np.abs(standard_vi.values - mc_result.values)
comparison_mc = pd.DataFrame({
    "Method": ["Value Iteration", "Off-Policy Monte Carlo"],
    "Uses full model?": ["Yes", "No"],
    "Main unit of work": ["Sweeps/iterations", "Episodes"],
    "Count": [standard_vi.iterations, mc_result.episodes],
    "Runtime seconds": [standard_vi.elapsed_seconds, mc_result.elapsed_seconds],
    "Complexity": ["O(K × |S| × |A|)", "O(episodes × episode length)"],
})

display(comparison_mc)
print(f"Mean absolute difference from VI values: {np.mean(absolute_errors):.4f}")
print(f"Maximum absolute difference from VI values: {np.max(absolute_errors):.4f}")

,Method,Uses full model?,Main unit of work,Count,Runtime seconds,Complexity
0,Value Iteration,Yes,Sweeps/iterations,9,0.069229,O(K × |S| × |A|)
1,Off-Policy Monte Carlo,No,Episodes,5000,3.586793,O(episodes × episode length)


Mean absolute difference from VI values: 0.0639
Maximum absolute difference from VI values: 0.3047


## Problem 4 comparison comments

Value iteration is both faster and more precise on this small gridworld because it works directly from the transition model. Every action from every state can be evaluated analytically in a single sweep.

Off-policy Monte Carlo has no access to the model. It constructs value estimates purely from experience by sampling episodes. This makes it applicable to environments where a model is unavailable, but the estimates converge more slowly and carry more variance than their dynamic-programming counterparts.

Despite this, the Monte Carlo estimates remain close to the value-iteration solution. Residual differences stem from finite sampling and the stochastic nature of the behavior policy. Collecting more episodes would generally reduce these discrepancies further.

## Problem 4 talking points

**A. Key RL feature:** The defining mechanism is off-policy learning via weighted importance sampling. The separation between the exploratory behavior policy and the greedy target policy allows the agent to gather broad experience while still refining a near-optimal control strategy.

**B. Implementation/testing challenge:** The main practical difficulty was preventing runaway episodes. A purely random policy can wander indefinitely in a grid without reaching the goal, so an episode length cap was necessary to keep training tractable.

**C. Why this is reinforcement learning:** This is RL because the agent builds its understanding entirely through interaction with the environment. Reward signals, episode returns, policies, and value estimates replace any form of labelled supervision — the agent learns what to do by experiencing consequences of its own actions.

# Final summary

This notebook provides a complete solution to all four reinforcement learning problems in a single executable file. It models a robotic manipulation task as an MDP, manually derives two iterations of value updates for a 2x2 Gridworld, implements and compares synchronous and in-place value iteration for a 5x5 Gridworld, and applies off-policy Monte Carlo control with weighted importance sampling to the same environment. Throughout, the code uses object-oriented design, logs execution details to a file, and presents algorithm comparisons by runtime, sweep or episode count, and computational complexity.

# References

- Sutton, R. S., & Barto, A. G. (2018). *Reinforcement Learning: An Introduction* (2nd ed.). MIT Press.
- CSCN8020 course materials covering Markov Decision Processes, Dynamic Programming, and Monte Carlo Methods.
- Gymnasium-style environment interface: `reset`, `step`, state, action, reward, terminated, and truncated interaction pattern.